# shuttle_lines_scoring - training

Detects **landed** shuttlecocks on the court floor, for in/out scoring. Trained
on the `shuttle_lines_scoring` Roboflow export: 58 frames from `lines.mp4`,
411 hand-drawn boxes.

### This is a different problem from the flying-shuttle model

| | flying shuttle | landed shuttle (here) |
|---|---|---|
| median box | 13.1 px | **34.5 px** |
| boxes/image | 1.0 | **7.1** |
| gate decision | `p2` (8-16 px band) | **`stock`** (>= 16 px) |
| starting weights | `yolov8-p2.yaml`, random init | **`yolov8n.pt`, COCO-pretrained** |

The two differences that matter:

**No P2 head.** Measured median is 34.5 px (min 17.3, max 67.3), comfortably
above the 16 px threshold, so the stride-4 head buys nothing and costs speed.

**Pretrained weights, not scratch.** The flying-shuttle run had to start from
random init because `yolov8-p2.yaml` has no COCO checkpoint. Stock `yolov8n`
does, and with only 36 training images that is decisive - a pretrained backbone
already knows edges, texture and shape, so the run only has to learn what a
shuttlecock looks like rather than what an object is.

### Known weakness

Every frame comes from one static camera, so the model sees one court, one
lighting state and one viewpoint. It will score well here and transfer poorly
to the other three cameras. Judge it on the test split, not the valid mAP.

Runtime -> Change runtime type -> **T4 GPU** before running.

In [ ]:
!nvidia-smi
!pip -q install ultralytics==8.4.14

## Upload the dataset

Run this cell, then choose `shuttle_lines_scoring.yolov8.zip` from
`setup/datasets/`.

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
import glob, os, zipfile, yaml
from collections import Counter

ZIP = "shuttle_lines_scoring.yolov8.zip"
ROOT = "/content/shuttle_lines_yolo"
with zipfile.ZipFile(ZIP) as zf:
    zf.extractall(ROOT)

# Roboflow writes relative paths ("../train/images") that only resolve from the
# yaml's own directory. Absolute paths survive whatever cwd Ultralytics picks.
DATA = f"{ROOT}/data.yaml"
with open(DATA, "w") as fh:
    yaml.safe_dump({"path": ROOT, "train": "train/images", "val": "valid/images",
                    "test": "test/images", "nc": 1, "names": ["shuttlecock"]},
                   fh, sort_keys=False)
print(open(DATA).read())

## Refuse to spend GPU time on a leaked split

`lines.mp4` is a static camera, so consecutive frames are nearly identical -
the leakage risk here is higher than it was for the rally clips, not lower.
Frames were assigned to splits in whole 70-frame blocks, so the check is that
no block appears in two splits.

In [ ]:
BLOCK = 70

def blocks_of(split):
    out = set()
    for p in glob.glob(f"{ROOT}/{split}/images/*.jpg"):
        # Roboflow appends a hash: lines_000423_jpg.rf.<hash>.jpg
        frame = int(os.path.basename(p).split("_")[1])
        out.add(frame // BLOCK)
    return out

tr, va, te = blocks_of("train"), blocks_of("valid"), blocks_of("test")
assert not (tr & va), f"blocks shared by train and valid: {sorted(tr & va)}"
assert not (tr & te), f"blocks shared by train and test: {sorted(tr & te)}"
assert not (va & te), f"blocks shared by valid and test: {sorted(va & te)}"

counts = {s: len(glob.glob(f"{ROOT}/{s}/images/*.jpg")) for s in ("train", "valid", "test")}
assert counts == {"train": 36, "valid": 13, "test": 9}, counts

import cv2
sizes = Counter(cv2.imread(p).shape[:2] for p in glob.glob(f"{ROOT}/*/images/*.jpg"))
assert list(sizes) == [(720, 1280)], f"unexpected resolution: {sizes}"

boxes = sum(len([l for l in open(p).read().splitlines() if l.strip()])
            for p in glob.glob(f"{ROOT}/*/labels/*.txt"))
print("counts:", counts)
print("blocks:", {"train": sorted(tr), "valid": sorted(va), "test": sorted(te)})
print("boxes:", boxes, " resolution:", dict(sizes))
print("OK: no block spans two splits")

## Train

In [ ]:
from ultralytics import YOLO

IMGSZ = 1280   # native long side; at 640 a 34 px shuttle is still 17 px, so 640 also works
PROJECT = "/content/runs/shuttle_lines"   # absolute: a relative project nests under runs_dir

# Pretrained, NOT a yaml. With 36 training images the COCO backbone is doing
# most of the work; random init would need orders of magnitude more data.
model = YOLO("yolov8n.pt")

results = model.train(
    data=DATA,
    epochs=150,          # the p2 run's best was epoch 103/120 and still improving
    imgsz=IMGSZ,
    batch=8,
    workers=2,
    seed=0,
    deterministic=True,
    project=PROJECT,
    name="stock-n",
    patience=40,
    cache=False,
    # --- mono-specific augmentation ---
    hsv_h=0.0,     # no hue information exists in this footage
    hsv_s=0.0,     # no saturation information exists either
    scale=0.4,     # less brutal than for the 13 px flying shuttle, still below default
    flipud=0.0,    # shuttles land cork-down; upside-down is not a real view
    fliplr=0.5,    # the court is roughly symmetric
    mosaic=1.0,
    close_mosaic=15,
    plots=True,
    val=True,
)

## Evaluate on the held-out test split

`best.pt` is selected on `valid`, so its valid score is optimistic by
construction. `test` never touched checkpoint selection - on 9 images and 50
boxes, read it as a smoke test rather than a measurement.

In [ ]:
best = YOLO(f"{results.save_dir}/weights/best.pt")
m = best.val(data=DATA, split="test", imgsz=IMGSZ, plots=False)
print(f"test  P={m.box.mp:.3f}  R={m.box.mr:.3f}  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}")

## Download

Unzip into `setup/runs/shuttle_lines/` locally, then run it over the clip:

```
python scripts/feeder_court/detect_video.py --show --full     --weights runs/shuttle_lines/stock-n/weights/best.pt     --videos datasets/vid_source/clear_badminton_dataset_for_collab/lines.mp4     --max-side 90 --out runs/shuttle_lines/detect
```

`--max-side 90` because these boxes run to 67 px, well above the 60 px filter
tuned for the flying shuttle.

In [ ]:
import shutil
RUN = str(results.save_dir)
shutil.make_archive("/content/shuttle_lines_train", "zip", RUN)
print(sorted(os.listdir(RUN)))
files.download("/content/shuttle_lines_train.zip")